In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/dangkhoa2016/KerasHub-TranslateGemma-27B-IT-Kaggle-TPU-v5e8-Text-Vision.git"
ROOT="/kaggle/working/KerasHub-TranslateGemma-27B-IT-Kaggle-TPU-v5e8-Text-Vision"

if [[ -d "$ROOT/.git" ]]; then
  echo "Refreshing existing repository checkout..."
  git -C "$ROOT" fetch origin main
  git -C "$ROOT" reset --hard origin/main
else
  rm -rf "$ROOT"
  git clone --depth 1 "$REPO_URL" "$ROOT"
fi


# TranslateGemma 27B IT — Kaggle TPU v5e-8 — Text + Vision REST Server

**English:**

This notebook is the recommended Kaggle entry point for the public `v1.0.0` repository. Import it with **File → Import Notebook → GitHub**, search for `dangkhoa2016/KerasHub-TranslateGemma-27B-IT-Kaggle-TPU-v5e8-Text-Vision`, and select `notebooks/kaggle-tpu-v5e8-text-vision.ipynb`. Before running, enable **Internet**, select **TPU v5e-8 / `v5litepod-8`**, attach the Keras TranslateGemma model containing `translategemma_27b_it`, keep `RUN_TPU_VALIDATION=True`, then use **Restart Session → Run All**. The notebook kernel intentionally avoids JAX/Keras imports; model initialization happens only inside the TPU worker.

**Tiếng Việt:**

Notebook này là entry point Kaggle được khuyến nghị cho public repository `v1.0.0`. Import bằng **File → Import Notebook → GitHub**, tìm `dangkhoa2016/KerasHub-TranslateGemma-27B-IT-Kaggle-TPU-v5e8-Text-Vision` và chọn `notebooks/kaggle-tpu-v5e8-text-vision.ipynb`. Trước khi chạy, bật **Internet**, chọn **TPU v5e-8 / `v5litepod-8`**, attach Keras TranslateGemma model chứa `translategemma_27b_it`, giữ `RUN_TPU_VALIDATION=True`, rồi dùng **Restart Session → Run All**. Notebook kernel chủ ý không import JAX/Keras; model chỉ được khởi tạo bên trong TPU worker.


## 1. Use the official repository as the runtime source

**English:** GitHub Import gives Kaggle this notebook; the first code cell clones or hard-refreshes `main` from the official `dangkhoa2016` repository into `/kaggle/working`. It verifies the checkout, prints the exact Git HEAD, requires a clean tree, and defines `RUN_TPU_VALIDATION`. Keep it `True` for the real 8-TPU workflow; use `False` only for dependency/unit/static validation.

**Tiếng Việt:** GitHub Import đưa notebook này vào Kaggle; code cell đầu clone hoặc hard-refresh `main` từ official repository `dangkhoa2016` vào `/kaggle/working`. Cell xác minh checkout, in exact Git HEAD, yêu cầu tree sạch và định nghĩa `RUN_TPU_VALIDATION`. Giữ `True` cho workflow 8 TPU thật; chỉ dùng `False` cho dependency/unit/static validation.


In [ ]:
from pathlib import Path
import os
import subprocess

WORK = Path("/kaggle/working")
ROOT = WORK / "KerasHub-TranslateGemma-27B-IT-Kaggle-TPU-v5e8-Text-Vision"

# Set False to run only dependency/unit/static validation while TPU is disabled.
RUN_TPU_VALIDATION = True

required = [
    ROOT / ".git",
    ROOT / ".env.example",
    ROOT / "scripts/setup.sh",
    ROOT / "scripts/start.sh",
    ROOT / "scripts/wait_ready.py",
    ROOT / "clients/python/translategemma_client.py",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Incomplete Git checkout at /kaggle/working/KerasHub-TranslateGemma-27B-IT-Kaggle-TPU-v5e8-Text-Vision:\n- "
        + "\n- ".join(missing)
    )

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
status = subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True)
if status.strip():
    raise RuntimeError("Refusing to continue from a dirty Git checkout:\n" + status)
print("ROOT =", ROOT)
print("HEAD =", head)
print("Git status = clean")


## 2. Setup dependencies and validate the TPU runtime

**English:** Setup preserves Kaggle's existing JAX/JAXLIB stack. Existing `libtpu` is retained; if it is absent, the helper installs `libtpu==0.0.17` with `--no-deps`. With TPU validation enabled, `TPU_PREFLIGHT_MODE=required` makes exactly 8 TPU devices a hard gate.

**Tiếng Việt:** Setup giữ nguyên JAX/JAXLIB hiện có của Kaggle. `libtpu` đã tồn tại sẽ được giữ; nếu chưa có, helper cài `libtpu==0.0.17` bằng `--no-deps`. Khi bật TPU validation, `TPU_PREFLIGHT_MODE=required` biến đúng 8 TPU devices thành hard gate.


In [ ]:
subprocess.run(["bash", "-lc", "cp -n .env.example .env || true"], cwd=ROOT, check=True)
env = os.environ.copy()
env["INSTALL_PYTHON_DEPS"] = "auto"
env["TPU_PREFLIGHT_MODE"] = "required" if RUN_TPU_VALIDATION else "skip"
subprocess.run(["bash", "scripts/setup.sh"], cwd=ROOT, env=env, check=True)


## 3. Start the coordinator and the single TPU worker

**English:** The Flask application is served by one Waitress coordinator process and stays CPU-side. One spawned TPU worker owns the logical TranslateGemma 27B model and shards it across the complete 8-device ModelParallel mesh.

**Tiếng Việt:** Flask application được phục vụ bởi một Waitress coordinator process phía CPU. Một TPU worker được spawn sở hữu logical model TranslateGemma 27B và shard model trên toàn bộ ModelParallel mesh 8 thiết bị.


In [ ]:
if RUN_TPU_VALIDATION:
    subprocess.run(["bash", "scripts/stop_tunnel.sh"], cwd=ROOT, check=False)
    subprocess.run(["bash", "scripts/stop.sh"], cwd=ROOT, check=False)
    subprocess.run(["bash", "scripts/start.sh"], cwd=ROOT, check=True)
else:
    print("SKIP server start: RUN_TPU_VALIDATION=False")


## 4. Wait for real TPU readiness

**English:** Readiness must prove exactly **8 TPU devices** and ModelParallel mesh **`[1,8]`**. A `503` while the worker is loading or compiling is expected; readiness becomes `200` only after model initialization completes.

**Tiếng Việt:** Readiness phải chứng minh đúng **8 TPU devices** và ModelParallel mesh **`[1,8]`**. `503` khi worker đang loading hoặc compiling là bình thường; readiness chỉ thành `200` sau khi model initialization hoàn tất.


In [ ]:
if RUN_TPU_VALIDATION:
    try:
        subprocess.run([
            "python3", "scripts/wait_ready.py",
            "--base-url", "http://127.0.0.1:7860",
            "--api-key-file", str(ROOT / "data/api_key.txt"),
            "--timeout", "1800",
            "--expected-devices", "8",
            "--expected-mesh", "1,8",
            "--heartbeat", "30",
        ], cwd=ROOT, check=True)
    except subprocess.CalledProcessError:
        subprocess.run(["bash", "scripts/status.sh"], cwd=ROOT, check=False)
        subprocess.run(["tail", "-n", "200", str(ROOT / "log/server.stdout.log")], check=False)
        raise
else:
    print("SKIP TPU readiness: RUN_TPU_VALIDATION=False")


## 5. Inspect authenticated runtime information

**English:** Query `/info` through the Python client. The response exposes safe runtime metadata such as model, backend, accelerator, device count, mesh, and vision support without exposing credentials or local model filesystem paths.

**Tiếng Việt:** Query `/info` qua Python client. Response hiển thị safe runtime metadata như model, backend, accelerator, device count, mesh và vision support mà không lộ credentials hoặc local model filesystem paths.


In [ ]:
if RUN_TPU_VALIDATION:
    INFO_ENDPOINT = "/info"
    subprocess.run([
        "python3", "clients/python/translategemma_client.py",
        "--base-url", "http://127.0.0.1:7860",
        "--api-key-file", "data/api_key.txt",
        "info",
    ], cwd=ROOT, check=True)
else:
    print("SKIP runtime info: RUN_TPU_VALIDATION=False")


## 6. Run the text translation smoke test

**English:** Exercise the text endpoint on the same TPU worker. Cold compilation can return `202`; the client follows the normal `/result/<job_id>` polling flow until completion.

**Tiếng Việt:** Kiểm tra text endpoint trên cùng TPU worker. Cold compilation có thể trả `202`; client theo normal `/result/<job_id>` polling flow tới khi hoàn tất.


In [ ]:
if RUN_TPU_VALIDATION:
    subprocess.run(["bash", "scripts/test.sh"], cwd=ROOT, check=True)
else:
    print("SKIP text smoke test: RUN_TPU_VALIDATION=False")


## 7. Run the multipart vision smoke test

**English:** Send the included sample image through `multipart/form-data` to verify the multimodal text+vision path on the same 8-device model.

**Tiếng Việt:** Gửi sample image đi kèm bằng `multipart/form-data` để xác minh multimodal text+vision path trên cùng model 8 thiết bị.


In [ ]:
if RUN_TPU_VALIDATION:
    subprocess.run([
        "python3", "clients/python/translategemma_client.py",
        "--base-url", "http://127.0.0.1:7860",
        "--api-key-file", "data/api_key.txt",
        "image", "assets/sample-image-with-text.png",
        "--source-lang", "English",
        "--target-lang", "Vietnamese",
        "--max-new-tokens", "256",
        "--multipart",
    ], cwd=ROOT, check=True)
else:
    print("SKIP multipart vision smoke test: RUN_TPU_VALIDATION=False")


## 8. Optional Cloudflare Quick Tunnel

**English:** The tunnel is disabled by default. Enable it only when temporary remote access is needed, and keep API authentication enabled whenever the service is exposed outside localhost.

**Tiếng Việt:** Tunnel mặc định tắt. Chỉ bật khi cần remote access tạm thời và luôn giữ API authentication khi service được expose ngoài localhost.


In [ ]:
if RUN_TPU_VALIDATION:
    START_TUNNEL = False
    if START_TUNNEL:
        subprocess.run(["bash", "scripts/run_tunnel.sh"], cwd=ROOT, check=True)
    else:
        print("Tunnel skipped. Run: python3 scripts/demo_info.py")
else:
    print("SKIP tunnel: RUN_TPU_VALIDATION=False")


## 9. Final service status

**English:** Print the managed process and worker status after validation. At this point the text and multipart vision smoke tests should already have completed successfully.

**Tiếng Việt:** In managed process và worker status sau validation. Tại thời điểm này text và multipart vision smoke tests phải đã hoàn tất thành công.


In [ ]:
if RUN_TPU_VALIDATION:
    subprocess.run(["bash", "scripts/status.sh"], cwd=ROOT, check=False)
else:
    print("SKIP final server status: RUN_TPU_VALIDATION=False")
